# Dataset Recogna

Junta os dois arquivos do Hugging Face. O terceiro trecho repete o extrativo, então ele entra uma vez só.

No FakeRecogna, a classe 0 é notícia falsa e a 1 é verdadeira. No extrativo é o contrário: 1 é falsa e 0 é verdadeira. Os rótulos são gravados como `falso` e `verdadeiro` para os dois ficarem na mesma escala.

Quando a mesma URL está nos dois arquivos e o rótulo coincide, fica o texto do extrativo. O texto do FakeRecogna está lematizado.


In [ ]:
import pandas as pd

# Os arquivos são públicos. O aviso de login do Hugging Face não bloqueia a leitura.
fakerecogna = pd.read_csv("hf://datasets/recogna-nlp/FakeRecogna/FakeRecogna.csv")
extrativo = pd.read_csv("hf://datasets/recogna-nlp/fakerecogna2-extrativa/fakerecogna_extrativo.csv")

fakerecogna.shape, extrativo.shape

((11903, 8), (52800, 9))

In [7]:
def palavras(serie):
    return serie.fillna("").astype(str).str.split().str.len()


def sem_url_conflitante(tabela):
    """Tira a URL que aparece com os dois rótulos e as repetições da mesma URL."""
    com_url = tabela.dropna(subset=["url"])
    conflito = com_url.groupby("url")["rotulo"].nunique().gt(1)
    tabela = tabela.loc[~tabela["url"].isin(conflito[conflito].index)]
    com_url = tabela.dropna(subset=["url"]).drop_duplicates("url")
    sem_url = tabela.loc[tabela["url"].isna()].drop_duplicates("noticia")
    return pd.concat([com_url, sem_url], ignore_index=True)


def preparar(tabela, rotulo, origem):
    tabela = tabela.copy()
    tabela["titulo"] = tabela["Titulo"].astype(str).str.strip()
    tabela["noticia"] = tabela["Noticia"].astype(str).str.strip()
    tabela["categoria"] = tabela["Categoria"].astype(str).str.strip().str.lower()
    tabela["data"] = tabela["Data"]
    tabela["url"] = tabela["URL"].where(tabela["URL"].notna())
    tabela["rotulo"] = rotulo
    tabela["origem"] = origem
    tabela = tabela.loc[tabela["rotulo"].notna()]
    tabela = tabela.loc[palavras(tabela["noticia"]) >= 8]
    colunas = ["origem", "titulo", "noticia", "categoria", "data", "url", "rotulo"]
    return sem_url_conflitante(tabela[colunas])


# Classe 0 do primeiro arquivo é falsa. Label 1 do extrativo também é falsa.
parte_fakerecogna = preparar(
    fakerecogna,
    fakerecogna["Classe"].map({0: "falso", 1: "verdadeiro"}),
    "FakeRecogna",
)
parte_extrativo = preparar(
    extrativo,
    extrativo["Label"].map({1: "falso", 0: "verdadeiro"}),
    "FakeRecogna 2",
)

em_comum = parte_fakerecogna.dropna(subset=["url"]).merge(
    parte_extrativo.dropna(subset=["url"]),
    on="url",
    suffixes=("_a", "_b"),
)
urls_iguais = em_comum.loc[em_comum["rotulo_a"] == em_comum["rotulo_b"], "url"]
urls_diferentes = em_comum.loc[em_comum["rotulo_a"] != em_comum["rotulo_b"], "url"]

# URL repetida com o mesmo rótulo: o extrativo conserva a frase original.
parte_fakerecogna = parte_fakerecogna.loc[~parte_fakerecogna["url"].isin(urls_iguais)]
parte_fakerecogna = parte_fakerecogna.loc[~parte_fakerecogna["url"].isin(urls_diferentes)]
parte_extrativo = parte_extrativo.loc[~parte_extrativo["url"].isin(urls_diferentes)]

dataset = pd.concat([parte_fakerecogna, parte_extrativo], ignore_index=True)
dataset = dataset.drop_duplicates("noticia").reset_index(drop=True)
dataset.to_csv("dataset_recogna.csv", index=False)
dataset.groupby(["origem", "rotulo"]).size()


origem         rotulo    
FakeRecogna    falso           865
               verdadeiro     5561
FakeRecogna 2  falso         23659
               verdadeiro    26175
dtype: int64

In [8]:
print(dataset["rotulo"].value_counts())
print("linhas", len(dataset))


rotulo
verdadeiro    31736
falso         24524
Name: count, dtype: int64
linhas 56260


In [9]:
categorias_politica = ["política", "eleições", "eleições 2022", "políticas públicas"]
politica = dataset.loc[dataset["categoria"].isin(categorias_politica)].copy()
politica["categoria"] = "politica"
politica = politica.reset_index(drop=True)
politica["rotulo"].value_counts()

rotulo
verdadeiro    9110
falso         4385
Name: count, dtype: int64

In [10]:
import re
import unicodedata

def veiculo(url):
    if pd.isna(url):
        return "sem url"
    host = re.search(r"https?://([^/]+)", str(url))
    if host is None:
        return "sem url"
    return re.sub(r"^www\.", "", host.group(1))

MESES = {
    "janeiro": "01", "fevereiro": "02", "março": "03", "marco": "03",
    "abril": "04", "maio": "05", "junho": "06", "julho": "07",
    "agosto": "08", "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12",
}

def formatar_data(valor):
    if pd.isna(valor):
        return pd.NA
    texto = str(valor).strip().split("Atualizada")[0]
    texto = re.sub(r"\s+às\s+\d{1,2}:\d{2}.*", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+\d{1,2}h\d{2}.*", "", texto).strip()
    numerica = re.match(r"(\d{1,2})/(\d{1,2})/(\d{4})", texto)
    if numerica:
        dia, mes, ano = numerica.groups()
        return f"{ano}-{int(mes):02d}-{int(dia):02d}"
    por_extenso = re.match(r"(\d{1,2})\s+(?:de\s+)?([A-Za-zçÇ]+)\s+(?:de\s+)?(\d{4})", texto)
    if por_extenso:
        dia, mes_nome, ano = por_extenso.groups()
        mes = MESES.get(mes_nome.lower())
        if mes:
            return f"{ano}-{mes}-{int(dia):02d}"
    iso = re.match(r"(\d{4})-(\d{2})-(\d{2})", texto)
    if iso:
        return iso.group(0)
    return pd.NA

falso = politica.loc[politica["rotulo"].eq("falso")]
verdadeiro = politica.loc[politica["rotulo"].eq("verdadeiro")].copy()
verdadeiro["veiculo"] = verdadeiro["url"].map(veiculo)

n_alvo = len(falso)
contagem = verdadeiro.groupby("veiculo").size()
bruto = contagem / contagem.sum() * n_alvo
quota = bruto.astype(int)
faltam = n_alvo - int(quota.sum())
maiores_restos = (bruto - quota).sort_values(ascending=False)
quota.loc[maiores_restos.index[:faltam]] += 1

partes = [
    verdadeiro.loc[verdadeiro["veiculo"].eq(site)].sample(n=int(n), random_state=42)
    for site, n in quota.items()
    if n > 0
]
verdadeiro_eq = pd.concat(partes).drop(columns="veiculo")
balanceado = pd.concat([falso, verdadeiro_eq], ignore_index=True)
balanceado = balanceado.sample(frac=1, random_state=42).reset_index(drop=True)
balanceado["data"] = balanceado["data"].map(formatar_data)
balanceado["categoria"] = "politica"
balanceado["url"] = balanceado["url"].map(veiculo).replace("sem url", pd.NA)
balanceado["rotulo"] = balanceado["rotulo"].map({"falso": 0, "verdadeiro": 1})
balanceado["quantidade_palavras"] = palavras(balanceado["noticia"])
balanceado["quantidade_pontuacao"] = balanceado["noticia"].map(
    lambda texto: sum(unicodedata.category(caractere).startswith("P") for caractere in str(texto))
)
balanceado.to_csv("dataset_recogna_politica_balanceado.csv", index=False)
balanceado[["quantidade_palavras", "quantidade_pontuacao"]].describe()

rotulo
verdadeiro    4385
falso         4385
Name: count, dtype: int64

In [ ]:
from pysentimiento import create_analyzer

NOMES = {"POS": 0, "NEU": 1, "NEG": 2}


def pedacos(texto, tamanho=80):
    palavras_texto = str(texto).split()
    return [
        " ".join(palavras_texto[inicio : inicio + tamanho])
        for inicio in range(0, len(palavras_texto), tamanho)
    ] or [""]


# O modelo lê no máximo 128 tokens. Textos longos são divididos e a probabilidade é a média das partes.
analisador = create_analyzer(task="sentiment", lang="pt")
grupos = [pedacos(texto) for texto in balanceado["noticia"]]
previsoes = analisador.predict([trecho for grupo in grupos for trecho in grupo])
inicio = 0
sentimentos = []
probabilidades = []
for grupo in grupos:
    fatia = previsoes[inicio : inicio + len(grupo)]
    inicio += len(grupo)
    medias = {
        chave: sum(saida.probas[chave] for saida in fatia) / len(fatia)
        for chave in ("NEG", "NEU", "POS")
    }
    sentimentos.append(NOMES[max(medias, key=medias.get)])
    probabilidades.append([round(medias[chave], 3) for chave in ("NEG", "NEU", "POS")])

balanceado = balanceado.drop(columns=["emocao"], errors="ignore")
balanceado["sentimento"] = sentimentos
balanceado[["prob_negativo", "prob_neutro", "prob_positivo"]] = probabilidades
balanceado.to_csv("dataset_recogna_politica_balanceado.csv", index=False)
balanceado["sentimento"].value_counts()

In [ ]:
import language_tool_python

corretor = language_tool_python.LanguageTool("pt-BR")
corretor.enabled_rules = {"MORFOLOGIK_RULE_PT_BR"}
corretor.enabled_rules_only = True


def erros_ortograficos(texto):
    marcas = corretor.check(str(texto))
    return sum(marca.rule_id == "MORFOLOGIK_RULE_PT_BR" for marca in marcas)


balanceado["erros_ortograficos"] = balanceado["noticia"].map(erros_ortograficos)
balanceado.to_csv("dataset_recogna_politica_balanceado.csv", index=False)
balanceado["erros_ortograficos"].describe()

In [ ]:
import spacy

nlp = spacy.load("pt_core_news_sm", disable=["parser", "ner"])
classes = {
    "quantidade_adverbios": {"ADV"},
    "quantidade_substantivos": {"NOUN"},
    "quantidade_adjetivos": {"ADJ"},
    "quantidade_verbos": {"VERB", "AUX"},
    "quantidade_substantivos_proprios": {"PROPN"},
    "quantidade_pronomes": {"PRON"},
}
contagens = {coluna: [] for coluna in classes}
for doc in nlp.pipe(balanceado["noticia"].astype(str), batch_size=64):
    etiquetas = [token.pos_ for token in doc]
    for coluna, grupo in classes.items():
        contagens[coluna].append(sum(etiqueta in grupo for etiqueta in etiquetas))

for coluna, valores in contagens.items():
    balanceado[coluna] = valores
balanceado.to_csv("dataset_recogna_politica_balanceado.csv", index=False)
balanceado[list(classes)].describe()